# Traditional ML Synthetic Options Backtest

This notebook is the Quant Orchestrator refactor target for `legacy optimal_trader/notebook_traditional_ml_synthetic_options_backtest.ipynb` on branch `codex/initial-sync`.

The reusable custom backtest engine and top-k signal rules live under `quant_orchestrator.platforms.backtesting_frameworks.panel_weight`; the synthetic option comparison workflow lives under `quant_orchestrator.research_tools`.

In [ ]:
from __future__ import annotations

import os
import pandas as pd
from IPython.display import display

from quant_orchestrator.platforms.registry import registry
from quant_orchestrator.research_tools import SyntheticOptionsBacktestConfig, run_synthetic_options_backtest

registry.get("backtesting_framework", "panel_weight")

## Inputs

`bt_panel` must be a `DataFrame` indexed by `date, symbol` with at least `close`, `prob_buy`, `prob_short`, and the configured score/component columns. The model training and feature-family scoring flow should produce that panel before this notebook runs the synthetic options comparison.

In [ ]:
CONFIG = SyntheticOptionsBacktestConfig(
    score_col="prob_buy",
    component_threshold=0.50,
    top_k_values=(5, 10, 20, 40),
    strategy_variants=("classifier_prob", "momentum_21d"),
    baseline_lookback_days=21,
    tenor_days=60,
    option_pricing_mode=os.getenv("OPTION_PRICING_MODE", "real_quotes"),
    real_quote_entry_price_col="ask",
    real_quote_exit_price_col="bid",
    real_quote_fallback_to_synthetic=False,
    enforce_option_capacity=os.getenv("ENFORCE_OPTION_CAPACITY", "0").lower() in {"1", "true", "yes"},
    max_volume_participation=float(os.getenv("MAX_VOLUME_PARTICIPATION", "0.10")),
    max_open_interest_participation=float(os.getenv("MAX_OPEN_INTEREST_PARTICIPATION", "0.02")),
    realized_vol_window=21,
    vol_floor=0.15,
    vol_cap=0.80,
    iv_multiplier=1.0,
    premium_floor=0.25,
    option_buckets={
        "atm_option": {"long_strike_multiplier": 1.00, "short_strike_multiplier": 1.00},
        "otm_option": {"long_strike_multiplier": 1.05, "short_strike_multiplier": 0.95},
        "ditm_option": {"long_strike_multiplier": 0.90, "short_strike_multiplier": 1.10},
    },
    initial_balance=100000.0,
    fee_bps=5.0,
    slippage_bps=5.0,
)

panel_path = os.getenv("BT_PANEL_PATH", "").strip()
if panel_path:
    bt_panel = pd.read_parquet(panel_path) if panel_path.endswith((".parquet", ".pq")) else pd.read_csv(panel_path)
    if not isinstance(bt_panel.index, pd.MultiIndex):
        bt_panel["date"] = pd.to_datetime(bt_panel["date"], errors="coerce").dt.normalize()
        bt_panel["symbol"] = bt_panel["symbol"].astype(str).str.upper()
        bt_panel = bt_panel.dropna(subset=["date", "symbol"]).set_index(["date", "symbol"]).sort_index()
else:
    # Set this manually if not using BT_PANEL_PATH.
    bt_panel = None

In [ ]:
if bt_panel is None:
    raise RuntimeError("Set bt_panel before running the synthetic options comparison.")

result = run_synthetic_options_backtest(bt_panel, config=CONFIG)
summary_df = result.summary
yearly_summary_df = result.yearly_summary
variant_runs = result.variant_runs
synthetic_price_panels = result.synthetic_price_panels
real_quote_coverage_df = result.real_quote_coverage

display(summary_df)

In [ ]:
display(yearly_summary_df)

In [ ]:
if not real_quote_coverage_df.empty:
    display(
        real_quote_coverage_df.groupby(["strategy", "instrument", "top_k", "side"], as_index=False)[
            ["selected_segments", "fallback_segments", "quote_return_count", "fallback_return_count"]
        ].sum()
    )
else:
    display(real_quote_coverage_df)

In [ ]:
summary_df.sort_values("sharpe", ascending=False)